<a href="https://colab.research.google.com/github/Mohsindahri/Fine-Tuning-TinyLLaMA-on-WikiLarge-for-Text-Simplification/blob/main/Qwen_Qwen2_5_3B_Instruct_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


In [5]:
import pandas as pd
import torch
from datasets import load_dataset
from tqdm import tqdm

!pip install evaluate
import evaluate
!pip install textstat
import textstat

from nltk.tokenize import sent_tokenize
import nltk
nltk.download("punkt")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [7]:
from datasets import load_dataset

dataset = load_dataset("facebook/asset")

print(dataset)

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

simplification/validation-00000-of-00001(…):   0%|          | 0.00/885k [00:00<?, ?B/s]

simplification/test-00000-of-00001.parqu(…):   0%|          | 0.00/170k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/359 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 359
    })
})


In [9]:
from datasets import load_dataset

asset = load_dataset(
    "facebook/asset",
    "simplification"
)

NUM_SAMPLES = 100

test_data = asset["test"].select(range(NUM_SAMPLES))

print("Total Samples:", len(test_data))

print("\nFirst Sample:\n")

print("Original:")
print(test_data[0]["original"])

print("\nReference Simplifications:")
print(test_data[0]["simplifications"])

Total Samples: 100

First Sample:

Original:
One side of the armed conflicts is composed mainly of the Sudanese military and the Janjaweed, a Sudanese militia group recruited mostly from the Afro-Arab Abbala tribes of the northern Rizeigat region in Sudan.

Reference Simplifications:
['On one side of the conflicts are the Sudanese military and the Janjaweed, a Sudanese militia group.  They are mostly recruited from the Afro-Arab Abbala tribes.', 'One side of the armed conflicts is composed mainly of the Sudanese military and the Janjaweed. The Janjaweed are a Sudanese militia group recruited mostly from the Afro-Arab Abbala tribes of the northern Rizeigat region in Sudan.', 'One side of the armed conflicts is mainly the Sudanese military and the Janjaweed militia group.', 'One side of the war is made up of the Sudanese military and the Janjaweed, a Sudanese militia group from the Afro-Arab Abbala tribes of the northern Rizeigat region in Sudan.', 'One side of the war is mainly made up 

In [11]:
def generate(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_text_with_prompt = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Attempt to extract only the generated part by removing the input prompt string
    if generated_text_with_prompt.startswith(prompt_text):
        return generated_text_with_prompt[len(prompt_text):].strip()
    return generated_text_with_prompt.strip()

def zero_shot_prompt(text):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that simplifies text."},
        {"role": "user", "content": f"Simplify the following text: {text}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def few_shot_prompt(text):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that simplifies text."},
        {"role": "user", "content": "Simplify the following text: The quick brown fox jumps over the lazy dog."},
        {"role": "assistant", "content": "The fox jumps over the dog."},
        {"role": "user", "content": f"Simplify the following text: {text}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def cot_prompt(text):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that simplifies text. Think step-by-step and then provide the simplified text."},
        {"role": "user", "content": f"Simplify the following text: {text}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


zero_predictions = []
few_predictions = []
cot_predictions = []

sources = []
references = []

results_df = []

for i, sample in enumerate(test_data):

    source = sample["original"]
    reference = sample["simplifications"]

    # Generate outputs
    zero_output = generate(zero_shot_prompt(source))
    few_output = generate(few_shot_prompt(source))
    cot_output = generate(cot_prompt(source))

    # Store for evaluation
    sources.append(source)
    references.append(reference)

    zero_predictions.append(zero_output)
    few_predictions.append(few_output)
    cot_predictions.append(cot_output)

    # Store everything in a dataframe
    results_df.append({
        "Original": source,
        "Reference": reference[0],
        "Zero Shot": zero_output,
        "Few Shot": few_output,
        "Chain of Thought": cot_output
    })

    print(f"{i+1}/100 completed")

results_df = pd.DataFrame(results_df)

print("\nGeneration Complete!")

1/100 completed
2/100 completed
3/100 completed
4/100 completed
5/100 completed
6/100 completed
7/100 completed
8/100 completed
9/100 completed
10/100 completed
11/100 completed
12/100 completed
13/100 completed
14/100 completed
15/100 completed
16/100 completed
17/100 completed
18/100 completed
19/100 completed
20/100 completed
21/100 completed
22/100 completed
23/100 completed
24/100 completed
25/100 completed
26/100 completed
27/100 completed
28/100 completed
29/100 completed
30/100 completed
31/100 completed
32/100 completed
33/100 completed
34/100 completed
35/100 completed
36/100 completed
37/100 completed
38/100 completed
39/100 completed
40/100 completed
41/100 completed
42/100 completed
43/100 completed
44/100 completed
45/100 completed
46/100 completed
47/100 completed
48/100 completed
49/100 completed
50/100 completed
51/100 completed
52/100 completed
53/100 completed
54/100 completed
55/100 completed
56/100 completed
57/100 completed
58/100 completed
59/100 completed
60/100

In [12]:
results_df.head(10)

,Original,Reference,Zero Shot,Few Shot,Chain of Thought
0,One side of the armed conflicts is composed ma...,On one side of the conflicts are the Sudanese ...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
1,"Jeddah is the principal gateway to Mecca, Isla...",Muslims are required to visit Mecca once in th...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
2,The Great Dark Spot is thought to represent a ...,The dark spot on Ne;tune may be a hole in the ...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
3,"His next work, Saturday, follows an especially...",Next Saturday is a presentation of a successfu...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
4,"The tarantula, the trickster character, spun a...",The tarantula spun a black cord and attached i...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
5,"There he died six weeks later, on 13 January 888.",He died there six weeks later on 13 January 888.,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
6,They are culturally akin to the coastal people...,They are culturally similar to the coastal peo...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
7,"Since 2000, the recipient of the Kate Greenawa...","Since 2000, the recipient of the Kate Greenawa...",system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
8,"Following the drummers are dancers, who often ...","After the drummers are dancers, who often play...",system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...
9,The spacecraft consists of two main elements: ...,The spacecraft's two main elements are the NAS...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...,system\nYou are a helpful assistant that simpl...


In [13]:
results_df.to_csv(
    "qwen_generated_outputs.csv",
    index=False
)

print("Saved as qwen_generated_outputs.csv")

Saved as qwen_generated_outputs.csv


In [17]:
!pip install rouge_score bert-score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b65d488ad5b9ea4f85d0c530301e59d2ad9af0a16eacfab7a11d07dc67a174ff
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [18]:
from evaluate import load
import textstat
import numpy as np

sari = load("sari")
bleu = load("bleu")
rouge = load("rouge")
bertscore = load("bertscore")

In [20]:
def evaluate_predictions(predictions, prompt_name):

    # SARI
    sari_score = sari.compute(
        sources=sources,
        predictions=predictions,
        references=references
    )["sari"]

    # BLEU
    bleu_score = bleu.compute(
        predictions=predictions,
        references=[[r[0]] for r in references]
    )["bleu"]

    # ROUGE-L
    rouge_score = rouge.compute(
        predictions=predictions,
        references=[r[0] for r in references]
    )["rougeL"]

    # FKGL
    fkgl = np.mean([
        textstat.flesch_kincaid_grade(p)
        for p in predictions
    ])

    # BERTScore
    bert = bertscore.compute(
        predictions=predictions,
        references=[r[0] for r in references],
        lang="en"
    )

    bert_f1 = np.mean(bert["f1"])

    return {
        "Model": MODEL_NAME,
        "Prompt": prompt_name,
        "SARI": sari_score,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "FKGL": fkgl,
        "BERTScore": bert_f1
    }

In [21]:
results = []

results.append(
    evaluate_predictions(
        zero_predictions,
        "Zero Shot"
    )
)

results.append(
    evaluate_predictions(
        few_predictions,
        "Few Shot"
    )
)

results.append(
    evaluate_predictions(
        cot_predictions,
        "Chain of Thought"
    )
)

results = pd.DataFrame(results)

results

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Model,Prompt,SARI,BLEU,ROUGE-L,FKGL,BERTScore
0,Qwen/Qwen2.5-3B-Instruct,Zero Shot,57.510003,0.182965,0.378525,10.975406,0.883338
1,Qwen/Qwen2.5-3B-Instruct,Few Shot,56.849829,0.132470,0.292089,9.161550,0.865065
2,Qwen/Qwen2.5-3B-Instruct,Chain of Thought,56.295693,0.069035,0.177710,9.788470,0.850681


In [22]:
results.to_csv(
    "qwen_evaluation_results.csv",
    index=False
)

print(results)

                      Model            Prompt       SARI      BLEU   ROUGE-L  \
0  Qwen/Qwen2.5-3B-Instruct         Zero Shot  57.510003  0.182965  0.378525   
1  Qwen/Qwen2.5-3B-Instruct          Few Shot  56.849829  0.132470  0.292089   
2  Qwen/Qwen2.5-3B-Instruct  Chain of Thought  56.295693  0.069035  0.177710   

        FKGL  BERTScore  
0  10.975406   0.883338  
1   9.161550   0.865065  
2   9.788470   0.850681  
